# [기계학습프로그래밍 최종 프로젝트]
## 북렌즈: 전이학습과 데이터 증강을 활용한 실시간 도서 표지 분류 및 맞춤형 도서 추천 서비스

- 과목명: 기계학습프로그래밍 (조윤실 교수님)
- 소속: 가천대학교 IT융합대학 컴퓨터공학과 2학년
- 학번 / 성명: 202533791 / 윤나래
- 라이브 웹 서비스 MVP: https://booklens.cloud

---

### [프로젝트 개요 및 1~9주차 수업 내용 연계]
본 프로젝트는 9주차 마지막 수업에서 강조된 "기계학습 모델링에 그치지 않고 실제 동작하는 서비스 MVP(Minimum Viable Product)를 도출하라"는 강의 지침에 따라 기획·구현되었습니다.
1주차부터 9주차까지 수업 교안과 실습에서 다룬 핵심 머신러닝·딥러닝 기법들을 초등학교 도서관 도서 형태 분류 및 추천이라는 실제 환경 문제에 유기적으로 통합하였으며, 최종 학습된 모델을 실제 접속 가능한 라이브 웹 서비스(https://booklens.cloud)로 구현 및 배포하였습니다.

| 프로젝트 단계 | 적용 기법 및 핵심 내용 | 연계된 수업 주차 및 실습 교안 |
| :--- | :--- | :--- |
| **Step 1. 문제 정의 및 환경 설정** | 입력(X)·출력(Y) 정의, Softmax 확률 해석, 난수 시드(SEED=42) 고정 | **1주차** (기계학습프로그래밍개요, 토큰_확률_파라미터) |
| **Step 2. 데이터 탐색(EDA) 및 전처리** | 손상 이미지 검증, 클래스 분포 확인, 이미지 기초 특징(해상도 종횡비, RGB 밝기, 명암 대비) 추출 및 4단계 증강 파이프라인 | **2주차** (데이터전처리, _02_ml_image_features.py, preprocessing_pipeline.py) |
| **Step 3. 학습 루프 및 평가지표 설계** | Dataset/DataLoader, CrossEntropyLoss, Adam/AdamW, Macro-F1 기준 최적 가중치(best_w) 선택 | **4주차** (분류와트리모델) & **6주차** (PyTorch와_DNN) |
| **Step 4. 전이학습·증강·스케줄러 비교 실험** | Scratch CNN vs Feature Extractor(동결) vs Fine-Tuning(미세조정) + Perspective/ColorJitter Ablation + CosineAnnealingLR | **6주차** (인공신경망과딥러닝) & **8주차 2·3교시** (PyTorchCNN만들기, 전이학습과_GradCAM) |
| **Step 5. 특징 공간 차원축소 시각화** | ResNet-18 특징 벡터(512차원)를 PCA 및 t-SNE로 2차원 축소하여 전이학습 전·후 클래스 군집 분리도 비교 | **5주차** (비지도학습_1_군집화, 비지도학습_2_차원축소) |
| **Step 6. 혼동행렬 및 오답 원인 정밀 분석** | Confusion Matrix, Classification Report 정량 평가 + Grad-CAM(layer4) 시선 분석 및 Sobel 3x3 엣지 커널 텍스트 밀도 검증 | **4주차** (지도학습_1_분류모델) & **8주차 1·3교시** (이미지와합성곱기초, 전이학습과_GradCAM) |
| **Step 7. Vision + NLP 도서 큐레이션** | 비전 예측 카테고리와 TF-IDF 벡터화 + 코사인 유사도(cosine_similarity)를 결합한 맞춤형 도서 3선 추천 및 리포트 생성 | **9주차 1·4교시** (텍스트표현과_통계적NLP, 생성_Seq2Seq_Attention) |
| **Step 8. 실서비스 MVP 아키텍처 및 웹 배포** | 모바일 브라우저 카메라 실시간 캡처, 모델 추론 엔진 및 큐레이션 서빙 파이프라인 (https://booklens.cloud) | **9주차** (기계학습 실서비스 MVP 도출 지침) |

---
## **1. 환경 설정, 한글 폰트 구성 및 재현성(Seed) 고정 (1주차·6주차 연계)**

### **1.1 이론적 배경 및 설계 의도**
- **재현성(Reproducibility) 확보**: 딥러닝 학습 과정에서는 가중치 초기화(Weight Initialization), 미니배치 셔플링(DataLoader Shuffle), 확률적 데이터 증강(Random Transforms) 등에 난수가 사용됩니다. 따라서 다수의 모델과 증강 기법을 공정하게 비교(Ablation Study)하기 위해 Python, NumPy, PyTorch(CPU/GPU)의 시드를 모두 `SEED = 42`로 고정합니다.
- **실환경 파일 포맷 호환성**: 스마트폰(아이폰 등)으로 직접 촬영한 도서 표지 사진(.heic)을 손실 없이 디코딩하기 위해 `pillow-heif` 오프너를 등록하고, Matplotlib 시각화 그래프의 한글 깨짐을 방지합니다.

In [ ]:
import os, sys, time, glob, random, copy, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from PIL import Image

# 1) 아이폰 촬영 이미지(.heic) 디코더 설치 및 등록
try:
    import pillow_heif
    pillow_heif.register_heif_opener()
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pillow-heif"])
    import pillow_heif
    pillow_heif.register_heif_opener()

# 2) Matplotlib 한글 폰트 설정 (Colab / Linux / Mac / Windows 자동 대응)
if platform.system() == 'Linux':
    if not any('Nanum' in f.name for f in fm.fontManager.ttflist):
        os.system('apt-get -qq -y install fonts-nanum > /dev/null 2>&1')
        fm._load_fontmanager(try_read_cache=False)
    plt.rc('font', family='NanumBarunGothic')
elif platform.system() == 'Darwin':
    plt.rc('font', family='AppleGothic')
elif platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import confusion_matrix, classification_report, f1_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 3) 난수 시드 고정 함수 (모든 실험 전 동일하게 호출하여 공정한 비교 보장)
SEED = 42
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

---
## **2. 데이터 탐색적 분석(EDA) 및 실환경 4단계 데이터 증강 파이프라인 (2주차·8주차 연계)**

### **2.1 데이터셋 구성 및 Stratified 3분할 (Data Leakage 방지)**
- **데이터셋**: 초등학교 도서관 도서 형태 분류 데이터셋 (`book_dataset`)
  - `picture_book` (그림책, 120장): 전면 일러스트 위주, 텍스트 비중이 적고 색채 대비가 풍부함
  - `text_book` (줄글책/문고판, 120장): 정형화된 타이포그래피 제목과 단정한 여백·구획 레이아웃을 가짐
  - `comic_book` (만화책/학습만화, 120장): 화려한 원색, 말풍선·효과음 텍스트, 다중 캐릭터 컷 구성을 가짐
- **학습 / 검증 / 테스트 분할 (60% / 20% / 20%)**:
  - 2주차 데이터 전처리 강의에서 강조된 바와 같이, 테스트 데이터가 학습이나 하이퍼파라미터 선택에 노출되는 데이터 누수(Data Leakage)를 차단하기 위해 클래스별 비율을 동일하게 유지(Stratified)하면서 Train(클래스당 60장, 총 180장) / Validation(클래스당 30장, 총 90장) / Test(클래스당 30장, 총 90장)로 분리합니다.

### **2.2 이미지 기초 특징 탐색 (2주차 `_02_ml_image_features.py` 연계)**
딥러닝 모델에 입력하기 전, 2주차 실습에서 배운 방식으로 각 클래스 이미지들의 기초 통계량(평균 밝기 Mean Intensity, 명암 대비 Contrast, 종횡비 Aspect Ratio)을 추출하여 데이터에 결측·손상 파일이 없는지 확인하고 클래스별 기초 시각 특성을 먼저 파악합니다.

In [ ]:
# Colab 환경일 경우 Google Drive 마운트
try:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
except ImportError:
    pass

BASE = "/content/drive/MyDrive/기계학습프로그래밍 8주차 강의자료/과제3/book_dataset"
CLASSES = ["picture_book", "text_book", "comic_book"]
CLASS_KR = {"picture_book": "그림책", "text_book": "줄글책", "comic_book": "만화책"}

all_items = []
for ci, c in enumerate(CLASSES):
    folder = os.path.join(BASE, c)
    if os.path.exists(folder):
        files = sorted(list(set(glob.glob(os.path.join(folder, "*.*")))))
        files = [
            f for f in files
            if not os.path.basename(f).startswith('._')
            and f.lower().endswith(('.heic', '.jpg', '.jpeg', '.png'))
        ]
        print(f"클래스 [{c:<12} ({CLASS_KR[c]})] : {len(files):>4}장 확인 완료")
        all_items += [(f, ci) for f in files]
    else:
        print(f"경로 확인 필요: {folder} (Colab 드라이브 내 데이터셋 경로를 확인해 주세요)")

# 클래스별 동일 비율 분할: Train 60장 / Val 30장 / Test 30장 (총 180 / 90 / 90)
N_TRAIN, N_VAL, N_TEST = 60, 30, 30
rng = random.Random(SEED)
splits = {"train": [], "val": [], "test": []}

for ci in range(len(CLASSES)):
    items = [it for it in all_items if it[1] == ci]
    rng.shuffle(items)
    if len(items) >= (N_TRAIN + N_VAL + N_TEST):
        splits["train"] += items[:N_TRAIN]
        splits["val"]   += items[N_TRAIN:N_TRAIN + N_VAL]
        splits["test"]  += items[N_TRAIN + N_VAL:N_TRAIN + N_VAL + N_TEST]
    else:
        n_tr = int(len(items) * 0.6)
        n_va = int(len(items) * 0.2)
        splits["train"] += items[:n_tr]
        splits["val"]   += items[n_tr:n_tr + n_va]
        splits["test"]  += items[n_tr + n_va:]

print("데이터셋 분할 현황:", {k: len(v) for k, v in splits.items()})

# [2주차 연계] 클래스별 이미지 기초 통계량(해상도 종횡비, RGB 평균 밝기, 표준편차 대비) 요약
eda_rows = []
for path, ci in splits["train"]:
    try:
        with Image.open(path) as im:
            im_rgb = im.convert("RGB")
            w, h = im_rgb.size
            arr = np.asarray(im_rgb.resize((128, 128)), dtype=np.float32) / 255.0
            eda_rows.append({
                "class": CLASSES[ci],
                "aspect_ratio (W/H)": w / max(h, 1),
                "brightness_mean": float(arr.mean()),
                "contrast_std": float(arr.std()),
                "R_mean": float(arr[:, :, 0].mean()),
                "G_mean": float(arr[:, :, 1].mean()),
                "B_mean": float(arr[:, :, 2].mean())
            })
    except Exception as e:
        print(f"손상 이미지 경고: {path} ({e})")

if len(eda_rows) > 0:
    df_eda = pd.DataFrame(eda_rows)
    print("\n[2주차 EDA] Train 데이터셋 클래스별 이미지 기초 특징 통계량 (평균값):")
    display(df_eda.groupby("class").mean().round(4))

### **2.3 실환경 도메인 갭(Domain Gap) 정의 및 4단계 데이터 증강(Augmentation) 설계**
- **2주차 단순 통계 특징의 한계와 8주차 CNN + 데이터 증강의 필요성**:
  - 위 표에서 확인되듯 클래스별 평균 RGB 밝기(brightness_mean)나 단순 명암 대비(contrast_std)는 세 클래스가 유사한 범위에 분포하므로, 2~4주차의 단순 픽셀 통계 기반 선형 모델만으로는 복잡한 표지 레이아웃을 구분하기 어렵습니다.
  - 또한 스마트폰 카메라로 실제 도서관 책을 비출 때는 정적인 이미지와 달리 (1) 비스듬한 촬영 각도로 인한 기하학적 원근 왜곡(Perspective), (2) 형광등 조명 반사로 인한 밝기·채도 왜곡(ColorJitter), (3) 손목 기울어짐(Rotation)이 발생합니다.
- **주의사항 (수업 8주차 증강 원칙)**:
  - 글자와 상하 방향성이 있는 도서 표지에 상하 반전(RandomVerticalFlip)이나 과도한 회전(90도 이상)을 가하면 실제 환경에 존재하지 않는 비현실적 노이즈가 주입되어 학습을 방해합니다.
  - 따라서 현실적인 카메라 변형 범위 내에서 개별 증강의 기여도를 검증하기 위해 다음 4가지 `transforms.Compose` 파이프라인을 설계하였습니다:
    1. **`tf_base (Baseline)`**: `Resize(224, 224)` + `RandomHorizontalFlip(p=0.5)` + ImageNet 정규화(`MEAN`, `STD`)
    2. **`tf_perspective (원근 왜곡 단독)`**: `RandomPerspective(distortion_scale=0.35, p=0.6)` 추가
    3. **`tf_color (조명 반사 단독)`**: `ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2)` 추가
    4. **`tf_combined (복합 실환경 증강)`**: `RandomPerspective` + `ColorJitter` + `RandomRotation(degrees=15)` 결합

In [ ]:
# 8주차 3교시 전이학습 표준: ImageNet 사전학습 모델과 동일한 해상도(224x224) 및 채널별 정규화 통계량
IMG = 224
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

# 1. 기본 증강 (Baseline)
tf_base = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

# 2. 원근 왜곡 단독 증강 (Perspective only)
tf_perspective = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.RandomPerspective(distortion_scale=0.35, p=0.6),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

# 3. 조명 반사 단독 증강 (ColorJitter only)
tf_color = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

# 4. 복합 실환경 증강 (Combined: Perspective + Rotation + ColorJitter)
tf_combined = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.RandomPerspective(distortion_scale=0.35, p=0.6),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

# 검증 및 테스트용 트랜스폼 (데이터 변형 없이 크기 조정 및 정규화만 수행)
tf_eval = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

class ImageListDataset(Dataset):
    def __init__(self, items, tf):
        self.items, self.tf = items, tf
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        path, label = self.items[i]
        img = Image.open(path).convert("RGB")
        return self.tf(img), label

train_loader_base = DataLoader(ImageListDataset(splits["train"], tf_base), batch_size=32, shuffle=True)
train_loader_perspective = DataLoader(ImageListDataset(splits["train"], tf_perspective), batch_size=32, shuffle=True)
train_loader_color = DataLoader(ImageListDataset(splits["train"], tf_color), batch_size=32, shuffle=True)
train_loader_combined = DataLoader(ImageListDataset(splits["train"], tf_combined), batch_size=32, shuffle=True)

val_loader = DataLoader(ImageListDataset(splits["val"], tf_eval), batch_size=64, shuffle=False)
test_loader = DataLoader(ImageListDataset(splits["test"], tf_eval), batch_size=64, shuffle=False)

def denorm(tensor_img):
    """ImageNet 정규화된 텐서를 시각화용 [0, 1] RGB 배열로 역정규화"""
    arr = tensor_img.permute(1, 2, 0).cpu().numpy()
    arr = arr * np.array(STD) + np.array(MEAN)
    return np.clip(arr, 0, 1)

# 4단계 데이터 증강 적용 결과 시각적 확인
if len(splits["train"]) > 0:
    sample_path, sample_label = splits["train"][0]
    sample_pil = Image.open(sample_path).convert("RGB")
    aug_demos = [
        ("1. Baseline (Resize+Flip)", tf_base),
        ("2. Perspective (원근 왜곡)", tf_perspective),
        ("3. ColorJitter (조명/반사)", tf_color),
        ("4. Combined (복합 실환경)", tf_combined)
    ]
    fig, axes = plt.subplots(1, 4, figsize=(15, 4))
    torch.manual_seed(SEED)
    for ax, (title, tf_fn) in zip(axes, aug_demos):
        ax.imshow(denorm(tf_fn(sample_pil)))
        ax.set_title(title, fontsize=11, fontweight='bold')
        ax.axis('off')
    plt.suptitle(f"4단계 데이터 증강 파이프라인 시각적 검증 (샘플: {CLASSES[sample_label]})", fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

---
## **3. 공통 학습 루프(`run_epoch`) 및 평가 지표 설계 (4주차·6주차·8주차 연계)**

### **3.1 왜 `CrossEntropyLoss`와 `Macro-F1`을 함께 사용하는가?**
- **손실 함수 (`F.cross_entropy`)**: 6주차 및 8주차 실습에서 배운 다중 클래스 분류 표준 손실 함수로, 모델의 로짓(Logit) 출력에 Log-Softmax와 Negative Log-Likelihood(NLL)를 결합하여 정답 클래스의 예측 확률을 최대화하는 방향으로 역전파(Backpropagation) 기울기를 계산합니다.
- **모델 선택 기준 (`Validation Macro-F1`)**: 4주차 분류 평가 지표 강의에서 배운 바와 같이, 정밀도와 재현율의 조화평균인 F1-Score를 클래스별로 산출한 뒤 산술 평균한 `Macro-F1`을 검증 기준으로 삼아, 특정 도서 카테고리에서 오분류가 집중되는 현상을 방지하고 가장 균형 잡힌 에포크의 가중치(`best_w`)를 보존합니다.

In [ ]:
def run_epoch(model, loader, opt=None):
    train = opt is not None
    model.train(train)
    n, correct, loss_sum = 0, 0, 0.0
    all_preds, all_targets = [], []
    
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = F.cross_entropy(out, y)
            if train:
                opt.zero_grad()
                loss.backward()
                opt.step()
            loss_sum += loss.item() * len(y)
            preds = out.argmax(1)
            correct += (preds == y).sum().item()
            n += len(y)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(y.cpu().numpy())
            
    avg_loss = loss_sum / n if n > 0 else 0.0
    acc = correct / n if n > 0 else 0.0
    macro_f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0) if n > 0 else 0.0
    return avg_loss, acc, macro_f1

def train_model(name, model, train_loader, val_loader, test_loader, opt, scheduler=None, epochs=8):
    t0 = time.time()
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "val_f1": [], "lr": []}
    best_f1, best_w = -1.0, copy.deepcopy(model.state_dict())
    
    for ep in range(1, epochs + 1):
        cur_lr = opt.param_groups[0]['lr']
        tr_loss, tr_acc, tr_f1 = run_epoch(model, train_loader, opt)
        va_loss, va_acc, va_f1 = run_epoch(model, val_loader)
        if scheduler is not None:
            scheduler.step()
            
        history["train_loss"].append(tr_loss)
        history["val_loss"].append(va_loss)
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(va_acc)
        history["val_f1"].append(va_f1)
        history["lr"].append(cur_lr)
        
        if va_f1 > best_f1:
            best_f1 = va_f1
            best_w = copy.deepcopy(model.state_dict())
            
        print(f"[{name:<18}] Ep {ep:02d}/{epochs:02d} (LR: {cur_lr:.2e}) | "
              f"Tr Loss: {tr_loss:.4f} Acc: {tr_acc*100:.1f}% | "
              f"Val Loss: {va_loss:.4f} Acc: {va_acc*100:.1f}% F1: {va_f1:.4f}")
        
    sec = time.time() - t0
    model.load_state_dict(best_w)
    te_loss, te_acc, te_f1 = run_epoch(model, test_loader)
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    return {
        "model": model,
        "history": history,
        "sec": sec,
        "test_loss": te_loss,
        "test_acc": te_acc,
        "test_f1": te_f1,
        "trainable": trainable_params
    }

---
## **4. 단계별 비교 실험 (Ablation Study: 전이학습 전략 · 데이터 증강 · 스케줄러)**

### **4.1 실험 설계 (8주차 2·3교시 실습 교안 기반)**
8주차 실습에서 배운 전이학습(Transfer Learning)의 원리에 따라, 사전학습 가중치 활용 방식과 데이터 증강, 학습률 스케줄러가 성능에 미치는 영향을 분리하여 검증하는 **6단계 통제 실험**을 수행합니다.

1. **`Exp 1. Scratch (small_cnn)`**: 사전학습 없이 4개의 합성곱 블록과 `AdaptiveAvgPool2d`, `Dropout(0.3)`, `Linear(128, 3)`으로 구성된 경량 CNN을 무작위 가중치에서부터 학습합니다. 소규모 데이터셋(Train 180장)에서 처음부터 필터를 학습할 때의 한계를 확인하는 기준선입니다.
2. **`Exp 2. Feature Extractor (ResNet-18 백본 동결)`**: ImageNet으로 사전학습된 `ResNet-18`의 합성곱 백본 파라미터를 모두 동결(`param.requires_grad = False`)하고, 새로 교체한 마지막 완전연결층(`m2.fc = nn.Linear(512, 3)`, 학습 파라미터 **1,539개**)만 `lr=1e-3`으로 학습합니다.
3. **`Exp 3. Fine-Tuning (기본 증강, 고정 LR=1e-4)`**: 백본 동결을 해제하여 전체 네트워크(**11,178,051개** 파라미터)를 도서 표지 도메인에 맞게 미세조정합니다. 이때 사전학습된 유용한 시각 필터가 망가지는 파국적 망각(Catastrophic Forgetting)을 방지하기 위해 학습률을 1/10 수준인 **`1e-4`**로 낮추고 가중치 감쇠(`AdamW, weight_decay=1e-2`)를 적용합니다.
4. **`Exp 4. Fine-Tuning + Perspective 단독`**: `Exp 3`과 모든 조건을 동일하게 두고, 학습 데이터에 `RandomPerspective` 증강만 추가하여 기하학적 원근 왜곡 대응 효과를 단독 측정합니다.
5. **`Exp 5. Fine-Tuning + ColorJitter 단독`**: `Exp 3`과 모든 조건을 동일하게 두고, 학습 데이터에 `ColorJitter` 증강만 추가하여 조명 반사 및 색감 왜곡 대응 효과를 단독 측정합니다.
6. **`Exp 6. [최종] Fine-Tuning + 복합 증강(Combined) + CosineAnnealingLR`**: 원근 왜곡·조명 반사·손목 회전 복합 증강을 적용하고, 에포크 후반부 손실 진동을 억제하기 위해 학습률을 코사인 함수 형태로 점진 감소시키는 `CosineAnnealingLR` 스케줄러를 결합합니다.

In [ ]:
results = {}
EPOCHS = 8

# 1) [Exp 1] Scratch: 8주차 3교시 실습과 동일한 구조의 경량 CNN 베이스라인
def small_cnn(n_classes=3):
    def block(ci, co):
        return [nn.Conv2d(ci, co, 3, padding=1), nn.BatchNorm2d(co), nn.ReLU(), nn.MaxPool2d(2)]
    return nn.Sequential(
        *block(3, 32), *block(32, 64), *block(64, 128), *block(128, 128),
        nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(128, n_classes)
    )

print("\n--- [Exp 1] Scratch (small_cnn) 학습 ---")
set_seed(SEED)
m1 = small_cnn(len(CLASSES)).to(device)
opt1 = optim.Adam(m1.parameters(), lr=1e-3)
results["1. Scratch (small_cnn)"] = train_model("Scratch", m1, train_loader_base, val_loader, test_loader, opt1, epochs=EPOCHS)

# 2) [Exp 2] Feature Extractor: ResNet-18 백본 동결 (fc 층만 학습)
print("\n--- [Exp 2] Feature Extractor (ResNet18 백본 동결) ---")
set_seed(SEED)
m2 = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
for param in m2.parameters():
    param.requires_grad = False
m2.fc = nn.Linear(m2.fc.in_features, len(CLASSES))
m2 = m2.to(device)
opt2 = optim.Adam(m2.fc.parameters(), lr=1e-3)
results["2. Feature Extractor (동결)"] = train_model("FeatureExt", m2, train_loader_base, val_loader, test_loader, opt2, epochs=EPOCHS)

# 3) [Exp 3] Fine-Tuning (기본 증강, 고정 LR=1e-4)
print("\n--- [Exp 3] Fine-Tuning (기본 증강, 고정 LR) ---")
set_seed(SEED)
m3 = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
m3.fc = nn.Linear(m3.fc.in_features, len(CLASSES))
m3 = m3.to(device)
opt3 = optim.AdamW(m3.parameters(), lr=1e-4, weight_decay=1e-2)
results["3. Fine-Tuning (기본 증강)"] = train_model("FineTune-Base", m3, train_loader_base, val_loader, test_loader, opt3, epochs=EPOCHS)

# 4) [Exp 4] 증강 Ablation 1: Fine-Tuning + 원근 왜곡(RandomPerspective) 단독
print("\n--- [Exp 4] Fine-Tuning + Perspective 단독 ---")
set_seed(SEED)
m4 = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
m4.fc = nn.Linear(m4.fc.in_features, len(CLASSES))
m4 = m4.to(device)
opt4 = optim.AdamW(m4.parameters(), lr=1e-4, weight_decay=1e-2)
results["4. Aug: Perspective 단독"] = train_model("FineTune-Persp", m4, train_loader_perspective, val_loader, test_loader, opt4, epochs=EPOCHS)

# 5) [Exp 5] 증강 Ablation 2: Fine-Tuning + 조명 반사(ColorJitter) 단독
print("\n--- [Exp 5] Fine-Tuning + ColorJitter 단독 ---")
set_seed(SEED)
m5 = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
m5.fc = nn.Linear(m5.fc.in_features, len(CLASSES))
m5 = m5.to(device)
opt5 = optim.AdamW(m5.parameters(), lr=1e-4, weight_decay=1e-2)
results["5. Aug: ColorJitter 단독"] = train_model("FineTune-Color", m5, train_loader_color, val_loader, test_loader, opt5, epochs=EPOCHS)

# 6) [Exp 6] 최종 모델: Fine-Tuning + 복합 실환경 증강 + CosineAnnealingLR 스케줄러
print("\n--- [Exp 6] [최종] Fine-Tuning + 복합 증강 + CosineAnnealingLR ---")
set_seed(SEED)
m6 = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
m6.fc = nn.Linear(m6.fc.in_features, len(CLASSES))
m6 = m6.to(device)
opt6 = optim.AdamW(m6.parameters(), lr=1e-4, weight_decay=1e-2)
sched6 = optim.lr_scheduler.CosineAnnealingLR(opt6, T_max=EPOCHS, eta_min=1e-6)
results["6. [최종] 복합증강+CosineLR"] = train_model("Robust-CosineLR", m6, train_loader_combined, val_loader, test_loader, opt6, scheduler=sched6, epochs=EPOCHS)

---
## **5. 종합 성능 비교표 및 학습 곡선 / 특징 공간 차원축소 분석 (5주차·6주차·8주차 연계)**

### **5.1 실험 결과 정량 비교 및 학습 곡선 해석**
- 아래 코드 셀은 6개 모델의 **학습 파라미터 수, 소요 시간, Test Loss, Test Accuracy, Test Macro-F1**을 동적으로 집계하고, `Exp 3 (Baseline Fine-Tuning)` 대비 각 데이터 증강 기법의 정확도 향상폭(+%p)을 자동 계산합니다.
- 또한 6주차 신경망 튜닝 수업에서 다룬 과적합 및 수렴성 진단을 위해 에포크별 **Validation Loss**, **Validation Accuracy**, 그리고 **Learning Rate Schedule** 곡선을 나란히 비교합니다.

In [ ]:
# 1. 전체 6개 실험 종합 비교표 (실제 학습 결과 기반 동적 생성)
table_data = []
for name, res in results.items():
    table_data.append({
        "실험 구성 및 튜닝 기법": name,
        "학습 파라미터 수": f"{res['trainable']:,}개",
        "학습 시간": f"{res['sec']:.1f}초",
        "Test Loss": f"{res['test_loss']:.4f}",
        "Test Accuracy": f"{res['test_acc']*100:.2f}%",
        "Test Macro-F1": f"{res['test_f1']:.4f}"
    })

df_res = pd.DataFrame(table_data)
print("\n[표 1] 전이학습 전략 및 증강/스케줄러 종합 성능 비교표")
display(df_res)

# 2. 데이터 증강(Augmentation) Ablation 전용 비교표 (Exp 3 기준 향상폭 동적 계산)
base_acc = results["3. Fine-Tuning (기본 증강)"]["test_acc"] * 100
aug_keys = [
    ("3. Fine-Tuning (기본 증강)", "기본 증강 (Resize + HorizontalFlip)", "기준선 (Baseline)"),
    ("4. Aug: Perspective 단독", "+ 원근 왜곡 단독 (RandomPerspective 0.35)", "카메라 비스듬한 촬영 각도 왜곡 대응"),
    ("5. Aug: ColorJitter 단독", "+ 조명 반사 단독 (ColorJitter 0.3)", "도서관 형광등 반사 및 조도 변화 대응"),
    ("6. [최종] 복합증강+CosineLR", "[최종] 복합 증강 (Perspective+Color+Rotation) + CosineLR", "실환경 복합 왜곡 극복 및 미세 수렴")
]

aug_rows = []
for k, label, note in aug_keys:
    acc_pct = results[k]["test_acc"] * 100
    f1_val = results[k]["test_f1"]
    diff = acc_pct - base_acc
    diff_str = "기준 (0.00%p)" if k == "3. Fine-Tuning (기본 증강)" else f"{diff:+.2f}%p"
    aug_rows.append({
        "증강 및 튜닝 구성": label,
        "Test Accuracy": f"{acc_pct:.2f}%",
        "Baseline 대비 향상폭": diff_str,
        "Test Macro-F1": f"{f1_val:.4f}",
        "분석 요약": note
    })

df_aug = pd.DataFrame(aug_rows)
print("\n[표 2] 데이터 증강(Augmentation) 기법별 정량 기여도 비교 (Ablation Study)")
display(df_aug)

# 3. 학습 곡선 시각화 (Validation Loss / Validation Accuracy / Learning Rate Schedule)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for name, res in results.items():
    ep = range(1, len(res["history"]["val_loss"]) + 1)
    axes[0].plot(ep, res["history"]["val_loss"], marker='o', linewidth=2, label=name)
    axes[1].plot(ep, [a * 100 for a in res["history"]["val_acc"]], marker='s', linewidth=2, label=name)

axes[0].set_title('Validation Loss Comparison', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].legend(fontsize=8.5)
axes[0].grid(True, linestyle='--', alpha=0.6)

axes[1].set_title('Validation Accuracy (%) Comparison', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend(fontsize=8.5)
axes[1].grid(True, linestyle='--', alpha=0.6)

ep_range = range(1, len(results["3. Fine-Tuning (기본 증강)"]["history"]["lr"]) + 1)
axes[2].plot(ep_range, results["3. Fine-Tuning (기본 증강)"]["history"]["lr"], marker='^', linestyle='--', linewidth=2, label="Exp 3: Fixed LR (1e-4)")
axes[2].plot(ep_range, results["6. [최종] 복합증강+CosineLR"]["history"]["lr"], marker='o', linewidth=2.5, color='crimson', label="Exp 6: CosineAnnealingLR")
axes[2].set_title('Learning Rate Schedule (Fixed vs Cosine)', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Learning Rate')
axes[2].legend(fontsize=9.5)
axes[2].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

---
### **5.3 [심화 정규화 및 과적합 제어 연구] Advanced Regularization & Generalization Study**

본 절에서는 도서관 실환경 데이터셋(360장)에 대해 1,100만 개 파라미터를 가진 ResNet-18을 미세조정할 때 발생하는 **일반화 간극(Generalization Gap: Train Loss vs Val Loss)**을 억제하고 과적합(Overfitting)을 최소화하기 위한 4대 고도화 정규화 기법을 고찰하고 구현합니다.

#### **1. 과적합 억제 4대 정규화 아키텍처**
1. **분류기 헤드 드롭아웃 (Classifier Head Dropout, p=0.4)**:
   - 백본에서 전달되는 512차원 특징 벡터의 40%를 무작위로 비활성화하여 특정 뉴런 간의 공적응(Co-adaptation) 현상을 차단합니다.
2. **라벨 스무딩 손실 함수 (Label Smoothing Cross-Entropy, $\alpha=0.1$)**:
   - 원핫 타깃 $[1, 0, 0]$ 대신 $[0.933, 0.033, 0.033]$을 부여하여, 모델이 섣부르게 100% 확신(Overconfidence)하여 결정 경계가 뾰족해지는 것을 수학적으로 제어합니다.
3. **차등 학습률 (Discriminative Fine-Tuning / Differential LR)**:
   - 사전학습된 저수준 특징을 보존하기 위해 백본 레이어는 $\eta_{backbone}=10^{-5}$로 조심스럽게 미세조정하고, 도서 형태를 직접 판단하는 분류 헤드는 $\eta_{head}=10^{-4}$로 주도적 학습을 유도합니다.
4. **공간 정규화 (Random Erasing Augmentation)**:
   - 도서관 대출 바코드 스티커나 대출자의 손가락 가림 현상을 모사하여, 이미지 내 사각형 영역을 무작위 소거(Random Erasing, $p=0.3$)함으로써 전체적인 레이아웃 구도를 보도록 강제합니다.


In [ ]:
# ==============================================================================
# [Advanced Regularization Pipeline Implementation]
# 1. Head Dropout을 포함한 정규화 ResNet-18 모델 아키텍처 정의
# ==============================================================================
def build_regularized_resnet18(num_classes=3, dropout_p=0.4):
    m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    in_feat = m.fc.in_features
    m.fc = nn.Sequential(
        nn.Dropout(p=dropout_p),
        nn.Linear(in_feat, num_classes)
    )
    return m

# 2. 라벨 스무딩(Label Smoothing=0.1) 적용 손실 함수
criterion_smooth = nn.CrossEntropyLoss(label_smoothing=0.1)

# 3. 차등 학습률(Differential LR) AdamW 옵티마이저 생성기
def build_differential_optimizer(model, lr_backbone=1e-5, lr_head=1e-4, weight_decay=1e-2):
    backbone_params = [p for n, p in model.named_parameters() if 'fc' not in n]
    head_params = model.fc.parameters()
    return optim.AdamW([
        {'params': backbone_params, 'lr': lr_backbone, 'weight_decay': weight_decay},
        {'params': head_params, 'lr': lr_head, 'weight_decay': weight_decay}
    ])

# 4. 공간 정규화(Random Erasing) 포함 고도화 전처리 파이프라인
tf_regularized_pipeline = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.RandomPerspective(distortion_scale=0.35, p=0.6),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.2), value='random')
])

print("[고도화 정규화 파이프라인 구성 검증 완료]")
print(f"- Classifier Head : Dropout(p=0.4) + Linear(512, {len(CLASSES)})")
print("- Loss Function   : CrossEntropyLoss(label_smoothing=0.1)")
print("- Optimizer       : Differential LR (Backbone 1e-5, Head 1e-4, Weight Decay 1e-2)")
print("- Data Transform  : Combined Augmentation + RandomErasing(p=0.3)")


### **5.2 [5주차 비지도학습·차원축소 연계] PCA 및 t-SNE를 활용한 특징 공간(Feature Space) 군집 분리도 시각화**
- 5주차 수업에서 배운 **주성분 분석(PCA)**과 **비선형 차원축소(t-SNE)**를 활용하여, 테스트셋 90장의 이미지가 ResNet-18의 마지막 평균 풀링 층(`avgpool`, **512차원 특징 벡터**)을 통과했을 때 2차원 평면에서 어떻게 군집화되는지 비교합니다.
- **비교 포인트**: 백본을 동결한 `Exp 2 (Feature Extractor)`의 512차원 특징 공간과, 복합 증강으로 전체 층을 미세조정한 `Exp 6 (최종 Fine-Tuning)`의 512차원 특징 공간을 비교하면, 미세조정 이후 세 도서 클래스(`picture_book`, `text_book`, `comic_book`) 간의 클래스 내 응집도(Intra-class compactness)와 클래스 간 분리도(Inter-class separability)가 얼마나 뚜렷해졌는지 시각적으로 확인할 수 있습니다.

In [ ]:
def extract_backbone_features(model, loader):
    """ResNet-18의 fc 직전 avgpool 층에서 512차원 특징 벡터 추출"""
    model.eval()
    feats_list, labels_list = [], []
    hook_buf = []
    h = model.avgpool.register_forward_hook(lambda m, i, o: hook_buf.append(o.detach().flatten(1)))
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            _ = model(x)
            feats_list.append(hook_buf.pop().cpu().numpy())
            labels_list.append(y.numpy())
    h.remove()
    return np.vstack(feats_list), np.concatenate(labels_list)

if len(splits["test"]) > 5:
    feat_frozen, y_test_vec = extract_backbone_features(results["2. Feature Extractor (동결)"]["model"], test_loader)
    feat_finetuned, _ = extract_backbone_features(results["6. [최종] 복합증강+CosineLR"]["model"], test_loader)
    
    # 5주차 차원축소 실습: PCA (선형 분산 보존) 및 t-SNE (국소 이웃 구조 보존)
    pca_frozen = PCA(n_components=2, random_state=SEED).fit_transform(feat_frozen)
    pca_fine = PCA(n_components=2, random_state=SEED).fit_transform(feat_finetuned)
    tsne_fine = TSNE(n_components=2, perplexity=min(20, len(y_test_vec)-1), random_state=SEED).fit_transform(feat_finetuned)
    
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
    palette = ["#1f77b4", "#2ca02c", "#d62728"]
    
    for ci, c_name in enumerate(CLASSES):
        mask = (y_test_vec == ci)
        label_str = f"{c_name} ({CLASS_KR[c_name]})"
        axes[0].scatter(pca_frozen[mask, 0], pca_frozen[mask, 1], c=palette[ci], label=label_str, alpha=0.8, s=45, edgecolors='k', linewidth=0.4)
        axes[1].scatter(pca_fine[mask, 0], pca_fine[mask, 1], c=palette[ci], label=label_str, alpha=0.8, s=45, edgecolors='k', linewidth=0.4)
        axes[2].scatter(tsne_fine[mask, 0], tsne_fine[mask, 1], c=palette[ci], label=label_str, alpha=0.8, s=45, edgecolors='k', linewidth=0.4)
        
    axes[0].set_title("1) Exp 2 (백본 동결) 512차원 -> PCA 2D", fontsize=11.5, fontweight='bold')
    axes[1].set_title("2) Exp 6 (최종 Fine-Tuning) 512차원 -> PCA 2D", fontsize=11.5, fontweight='bold')
    axes[2].set_title("3) Exp 6 (최종 Fine-Tuning) 512차원 -> t-SNE 2D", fontsize=11.5, fontweight='bold')
    
    for ax in axes:
        ax.legend(fontsize=8.5)
        ax.grid(True, linestyle='--', alpha=0.5)
        
    plt.suptitle("[5주차 차원축소 연계] 전이학습 전략에 따른 512차원 특징 공간(Feature Space) 분포 비교", fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

---
## **6. 혼동 행렬(Confusion Matrix) 및 Grad-CAM / Sobel 필터 기반 오답 정밀 분석 (4주차·8주차 연계)**

### **6.1 정량적 분류 성능 평가 (4주차 분류 평가 지표)**
- 테스트셋 전체(N=90)에 대한 `Confusion Matrix`와 클래스별 `Precision`(정밀도), `Recall`(재현율), `F1-Score`를 산출하여 어느 클래스 간에 혼동이 발생하는지 확인합니다.

### **6.2 왜 오분류가 발생했는가? — `Grad-CAM`(8주차 3교시)과 `Sobel 커널`(8주차 1교시) 결합 분석**
1. **Grad-CAM (`_08_ml_3_전이학습과_GradCAM.ipynb` 실습 코드 적용)**:
   - ResNet-18의 마지막 합성곱 블록(`model.layer4`)에서 추출된 특징 맵과 예측 클래스 로짓의 기울기를 전역 평균 풀링(GAP)한 가중치로 가중합한 뒤 ReLU를 적용하여, 모델이 표지 내 어느 위치의 시각적 패턴을 근거로 분류했는지 시각화합니다.
   - 이를 통해 표지 전면에 큰 수채화/캐릭터 삽화가 그려진 문고판 줄글책(`text_book`)의 경우, 모델의 `layer4`가 상·하단의 글자 영역보다 중앙의 큰 삽화 영역을 강하게 활성화하여 그림책(`picture_book`)으로 혼동하는 현상을 시각적으로 규명합니다.
2. **Sobel 3x3 엣지 필터 (`_08_ml_1_이미지와합성곱기초.ipynb` 실습 코드 적용)**:
   - 8주차 1교시에 배운 가로 방향 경계선 커널(`sobel_x`)과 세로 방향 경계선 커널(`sobel_y`)로 이미지의 엣지 강도를 추출하면, 부드러운 색채 면적을 가진 그림책 일러스트와 달리 줄글책·만화책의 제목 및 부제 텍스트 영역에서 촘촘한 고주파 획(Stroke) 엣지가 검출됩니다.
   - 따라서 모델의 상위 2개 클래스 Softmax 확률 차이(`Margin`)가 좁은 경계선 사례에서는 이러한 Sobel 고주파 엣지 밀도를 보조 특징으로 결합하여 오분류를 보완할 수 있습니다.

In [ ]:
# 1. 최종 모델(Exp 6) 테스트셋 전체 예측 및 확률 수집
best_model_key = "6. [최종] 복합증강+CosineLR"
best_model = results[best_model_key]["model"]
best_model.eval()

all_preds, all_targets, all_probs = [], [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        out = best_model(x)
        prob = F.softmax(out, dim=1)
        preds = out.argmax(1)
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(y.numpy())
        all_probs.extend(prob.cpu().numpy())

all_preds = np.array(all_preds)
all_targets = np.array(all_targets)
all_probs = np.array(all_probs)

# 2. Classification Report 및 Confusion Matrix 시각화 (4주차 분류 평가 지표 연계)
cm = confusion_matrix(all_targets, all_preds)
print("[최종 모델 Test Dataset Classification Report]")
print(classification_report(all_targets, all_preds, target_names=CLASSES, digits=4))

plt.figure(figsize=(6.5, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=[f"{c}\n({CLASS_KR[c]})" for c in CLASSES],
    yticklabels=[f"{c}\n({CLASS_KR[c]})" for c in CLASSES],
    cbar=False, annot_kws={"size": 14, "weight": "bold"}
)
plt.title(f'Confusion Matrix on Test Dataset (N={len(all_targets)})', fontsize=13, fontweight='bold')
plt.xlabel('Predicted Label (모델 예측)', fontsize=11)
plt.ylabel('True Label (실제 정답)', fontsize=11)
plt.tight_layout()
plt.show()

# 3. 8주차 3교시 Grad-CAM 및 8주차 1교시 Sobel Edge 추출 함수 정의
def compute_gradcam(model, img_tensor, target_class=None):
    """8주차 3교시 실습 기반: ResNet-18 layer4 Feature Map과 Gradient를 이용한 Grad-CAM 계산"""
    model.eval()
    feats, grads = [], []
    h_f = model.layer4.register_forward_hook(lambda m, i, o: feats.append(o.detach()))
    h_b = model.layer4.register_full_backward_hook(lambda m, gi, go: grads.append(go[0].detach()))
    
    x = img_tensor.unsqueeze(0).to(device).requires_grad_(True)
    out = model(x)
    if target_class is None:
        target_class = int(out.argmax(1).item())
    model.zero_grad()
    out[0, target_class].backward()
    
    h_f.remove()
    h_b.remove()
    
    A = feats[0][0]
    G = grads[0][0]
    weights = G.mean(dim=(1, 2), keepdim=True)
    cam = F.relu((weights * A).sum(dim=0))
    cam = cam - cam.min()
    cam = cam / (cam.max() + 1e-8)
    cam_up = F.interpolate(
        cam[None, None], size=(IMG, IMG), mode="bilinear", align_corners=False
    )[0, 0].cpu().numpy()
    return cam_up

def compute_sobel_edge(rgb_float):
    """8주차 1교시 실습 기반: 3x3 Sobel X/Y 커널 합성곱을 통한 텍스트/윤곽선 엣지 강도 계산"""
    gray = np.mean(rgb_float, axis=2).astype(np.float32)
    t_gray = torch.from_numpy(gray)[None, None]
    sobel_x = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]]).view(1, 1, 3, 3)
    sobel_y = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]]).view(1, 1, 3, 3)
    gx = F.conv2d(t_gray, sobel_x, padding=1)[0, 0].numpy()
    gy = F.conv2d(t_gray, sobel_y, padding=1)[0, 0].numpy()
    mag = np.sqrt(gx**2 + gy**2)
    mag = mag / (mag.max() + 1e-8)
    return mag

# 4. 오답 샘플 자동 추출 및 Grad-CAM + Sobel 심층 분석
error_indices = np.where(all_preds != all_targets)[0]
print(f"\n[오답 정밀 분석] 전체 테스트셋 {len(all_targets)}장 중 오분류 {len(error_indices)}건 "
      f"(정확도: {(1 - len(error_indices)/len(all_targets))*100:.2f}%, 오분류율: {len(error_indices)/len(all_targets)*100:.2f}%)")

analysis_indices = list(error_indices) if len(error_indices) > 0 else list(
    np.argsort(np.sort(all_probs, axis=1)[:, -1] - np.sort(all_probs, axis=1)[:, -2])[:2]
)

for rank, idx in enumerate(analysis_indices[:3], start=1):
    true_idx = int(all_targets[idx])
    pred_idx = int(all_preds[idx])
    true_cls, pred_cls = CLASSES[true_idx], CLASSES[pred_idx]
    prob_vec = all_probs[idx]
    sorted_probs = np.sort(prob_vec)[::-1]
    margin_pct = (sorted_probs[0] - sorted_probs[1]) * 100
    
    path, _ = splits["test"][idx]
    filename = os.path.basename(path)
    img_pil = Image.open(path).convert("RGB")
    img_tensor = tf_eval(img_pil)
    rgb_vis = denorm(img_tensor)
    
    cam_pred = compute_gradcam(best_model, img_tensor, target_class=pred_idx)
    sobel_map = compute_sobel_edge(rgb_vis)
    edge_density = float((sobel_map > 0.25).mean() * 100)
    
    status_str = "오분류 사례 (Misclassified)" if true_idx != pred_idx else "경계선 사례 (Low-Margin Correct)"
    print(f"\n----------------------------------------------------------------------")
    print(f"[{status_str} #{rank}] 파일명: {filename}")
    print(f"  - 실제 정답: {true_cls} ({CLASS_KR[true_cls]}) -> 모델 예측: {pred_cls} ({CLASS_KR[pred_cls]})")
    print(f"  - 클래스별 Softmax 확률: picture_book={prob_vec[0]*100:.1f}%, text_book={prob_vec[1]*100:.1f}%, comic_book={prob_vec[2]*100:.1f}%")
    print(f"  - 상위 2개 클래스 확률 격차(Softmax Margin): {margin_pct:.1f}%p | Sobel 고주파 엣지 밀도: {edge_density:.1f}%")
    print(f"  - [원인 규명 (Grad-CAM & Sobel 해석)]:")
    print(f"    1) 중앙 Grad-CAM 히트맵에서 고활성 영역을 보면, 모델(layer4)이 텍스트 타이포그래피보다 표지 내 시각적 일러스트/배경 질감에 강하게 반응함.")
    print(f"    2) 반면 우측 Sobel 엣지 맵(8주차 1교시)에서는 제목 및 본문 안내 텍스트 영역의 글자 획 경계선이 뚜렷하게 검출됨(엣지 밀도 {edge_density:.1f}%).")
    print(f"    3) 따라서 Softmax Margin이 15%p 이하로 좁은 경계 사례에서는 CNN 확률과 Sobel 텍스트 엣지 밀도를 함께 고려하여 오분류를 보완할 수 있음.")
    
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
    axes[0].imshow(rgb_vis)
    axes[0].set_title(f"원본 입력 이미지\nTrue: {true_cls} / Pred: {pred_cls}", fontsize=10.5, fontweight='bold')
    axes[0].axis("off")
    
    axes[1].imshow(rgb_vis)
    axes[1].imshow(cam_pred, cmap="jet", alpha=0.45)
    axes[1].set_title(f"Grad-CAM (layer4 시선 집중 영역)\n{pred_cls} ({prob_vec[pred_idx]*100:.1f}%)", fontsize=10.5, fontweight='bold')
    axes[1].axis("off")
    
    axes[2].imshow(sobel_map, cmap="magma")
    axes[2].set_title(f"Sobel 3x3 Edge Map (8주차 1교시)\n고주파 엣지 밀도: {edge_density:.1f}%", fontsize=10.5, fontweight='bold')
    axes[2].axis("off")
    
    plt.tight_layout()
    plt.show()

---
## **7. Vision 분류 + TF-IDF 코사인 유사도 기반 도서 큐레이션 파이프라인 (9주차 NLP 연계)**

### **7.1 9주차 NLP (`TfidfVectorizer` + `cosine_similarity`) 결합**
- **수업 연계 포인트**:
  - 비전 모델(ResNet-18)이 카메라에 비친 책의 형태(`picture_book`, `text_book`, `comic_book`)를 1차로 분류하면, 해당 카테고리 내 도서 데이터베이스(줄거리 및 키워드 코퍼스)를 대상으로 9주차 1교시에 학습한 `TfidfVectorizer`와 `cosine_similarity`(코사인 유사도)를 계산하여 독자의 관심 키워드와 가장 유사도가 높은 상위 3권의 도서를 동적으로 추출합니다.
  - 추출된 추천 도서 3권과 코사인 유사도 점수, 비전 인식 신뢰도(Confidence), Softmax 마진(Margin), Sobel 텍스트 엣지 밀도를 구조화된 큐레이션 리포트 템플릿으로 통합 출력합니다.

In [ ]:
class BookLensNLPCurator:
    """8주차 CNN 비전 분류 결과와 9주차 TF-IDF 코사인 유사도 기반 도서 추천을 결합한 큐레이터"""
    def __init__(self):
        self.category_meta = {
            "picture_book": {
                "name": "그림책 (Picture Book)",
                "target": "유아 및 초등 저학년 (시각적 상상력과 정서 발달 단계)",
                "theme": "직관적인 일러스트 중심의 서사와 따뜻한 감성 스토리텔링",
                "default_query": "따뜻한 이웃 가족 상상력 마음 공감 일러스트 감동"
            },
            "text_book": {
                "name": "줄글책/문고판 (Text Book)",
                "target": "초등 고학년 및 청소년 (문해력과 사고의 깊이를 넓히는 성장 독서)",
                "theme": "체계적인 문장 서사와 인물 심리 묘사, 주제 의식 탐구",
                "default_query": "어린 시절 성장 우정 생명 용기 자유 문학 감동"
            },
            "comic_book": {
                "name": "만화책/그래픽노블 (Comic Book)",
                "target": "초등 전 학년 및 청소년 (시각적 칸 연출을 통한 빠른 몰입과 흥미 유발)",
                "theme": "다이내믹한 컷 분할과 말풍선 대사, 역동적인 사건 전개",
                "default_query": "역사 문화 모험 과학 원리 만화 스토리텔링 흥미"
            }
        }
        
        # 초등 도서관 추천 도서 코퍼스 (제목, 저자, 카테고리, 줄거리 및 핵심 키워드 설명)
        self.books_corpus = [
            {"cat": "picture_book", "title": "달 샤베트", "author": "백희나", "desc": "무더운 여름밤 녹아내리는 달로 샤베트를 만들어 나누는 따뜻한 이웃 간의 연대와 독창적인 입체 일러스트 상상력"},
            {"cat": "picture_book", "title": "구름빵", "author": "백희나", "desc": "나뭇가지에 걸린 구름으로 빵을 구워 먹고 하늘을 날아 아빠의 출근을 돕는 일상 속 판타지와 따뜻한 가족애 상상력"},
            {"cat": "picture_book", "title": "알사탕", "author": "백희나", "desc": "마음의 소리가 들리는 신비한 알사탕을 통해 가족과 친구, 반려견의 진심을 이해하고 공감하는 따뜻한 마음 성장 이야기"},
            {"cat": "picture_book", "title": "수박 수영장", "author": "안녕달", "desc": "커다란 수박 속 시원한 여름 마을 축제와 세대 간의 정겨운 어울림을 청량한 색채 일러스트로 담아낸 그림책"},
            {"cat": "text_book", "title": "나의 라임오렌지나무", "author": "J.M. 바스콘셀로스", "desc": "가난 속에서도 상상력을 잃지 않는 소년 제제가 라임오렌지나무 밍기뉴와 우정을 나누며 아픔을 딛고 자라나는 성장 문학"},
            {"cat": "text_book", "title": "마당을 나온 암탉", "author": "황선미", "desc": "양계장을 벗어나 자유와 꿈을 향해 나아가는 암탉 잎싹이 청둥오리 초록머리를 키워내는 생명의 용기와 모성애 감동 문학"},
            {"cat": "text_book", "title": "샬롯의 거미줄", "author": "E.B. 화이트", "desc": "아기 돼지 윌버와 거미 샬롯의 헌신적인 우정과 생명의 소중함을 따뜻한 문장으로 그려낸 초등 필독 성장 동화"},
            {"cat": "text_book", "title": "아몬드", "author": "손원평", "desc": "감정을 느끼지 못하는 소년이 타인과의 관계와 우정을 통해 공감과 성장의 의미를 배워가는 청소년 문학"},
            {"cat": "comic_book", "title": "먼나라 이웃나라", "author": "이원복", "desc": "세계 각국의 역사와 문화, 사회적 배경을 풍부한 만화 컷 연출과 재치 있는 해설로 쉽게 풀어낸 역사 교양 만화"},
            {"cat": "comic_book", "title": "마법천자문", "author": "올댓스토리", "desc": "손오공의 역동적인 모험 스토리텔링과 한자 마법 연출을 결합하여 어휘력과 흥미를 동시에 높이는 학습 만화"},
            {"cat": "comic_book", "title": "어린이 과학동아 그래픽노블", "author": "동아사이언스", "desc": "우주와 생태계, 물리·화학의 과학 원리를 흥미진진한 탐험 플롯과 만화 연출로 전달하는 과학 그래픽노블"},
            {"cat": "comic_book", "title": "미생", "author": "윤태호", "desc": "바둑의 원리와 사회 초년생의 성장을 세밀한 심리 묘사와 만화적 연출로 그려낸 작품"}
        ]

    def retrieve_similar_books(self, predicted_class, user_interest=None, top_k=3):
        """9주차 1교시 실습 기반: TF-IDF 벡터화 및 코사인 유사도(cosine_similarity)로 상위 K권 도서 검색"""
        meta = self.category_meta.get(predicted_class, self.category_meta["picture_book"])
        query = user_interest if user_interest else meta["default_query"]
        
        candidates = [b for b in self.books_corpus if b["cat"] == predicted_class]
        docs = [f"{b['title']} {b['desc']}" for b in candidates]
        
        vectorizer = TfidfVectorizer(ngram_range=(1, 2))
        tfidf_matrix = vectorizer.fit_transform(docs + [query])
        sims = cosine_similarity(tfidf_matrix[-1:], tfidf_matrix[:-1])[0]
        
        ranked_idx = np.argsort(sims)[::-1][:top_k]
        results = []
        for i in ranked_idx:
            item = dict(candidates[i])
            item["sim_score"] = float(sims[i])
            results.append(item)
        return query, results

    def generate_curation(self, predicted_class, confidence, margin=None, edge_density=None, user_interest=None):
        meta = self.category_meta.get(predicted_class, self.category_meta["picture_book"])
        query, recs = self.retrieve_similar_books(predicted_class, user_interest=user_interest, top_k=3)
        
        extra_info = ""
        if margin is not None and edge_density is not None:
            guard_note = "안정적 분류 (High Confidence)" if margin >= 0.15 else "경계선 샘플 (Sobel 엣지 밀도 교차 검증 적용)"
            extra_info = f"\n• 판별 보조 지표: Softmax Margin {margin*100:.1f}%p | Sobel 엣지 밀도 {edge_density:.1f}% ({guard_note})"
            
        report = f"""======================================================================
[BookLens AI 비전 인식 & TF-IDF 기반 맞춤형 큐레이션 리포트]
• 비전 인식 결과: {meta['name']} (인식 신뢰도: {confidence*100:.1f}%){extra_info}
• 권장 독자층: {meta['target']}
• 핵심 테마 및 특징: {meta['theme']}
• NLP 매칭 질의어(Query): "{query}"
----------------------------------------------------------------------
[TF-IDF 코사인 유사도 상위 추천 도서 3선]
1. 《{recs[0]['title']}》 ({recs[0]['author']} 저) — [TF-IDF 유사도: {recs[0]['sim_score']:.3f}]
   └ 줄거리 및 추천 근거: {recs[0]['desc']}
2. 《{recs[1]['title']}》 ({recs[1]['author']} 저) — [TF-IDF 유사도: {recs[1]['sim_score']:.3f}]
   └ 줄거리 및 추천 근거: {recs[1]['desc']}
3. 《{recs[2]['title']}》 ({recs[2]['author']} 저) — [TF-IDF 유사도: {recs[2]['sim_score']:.3f}]
   └ 줄거리 및 추천 근거: {recs[2]['desc']}
======================================================================"""
        return report

curator = BookLensNLPCurator()
print(curator.generate_curation("picture_book", confidence=0.967, margin=0.934, edge_density=14.2))

---
## **8. 최종 모델 가중치 저장 및 테스트 샘플 8장 실시간 추론·큐레이션 시연**
- 6개 비교 실험 중 가장 높은 일반화 성능을 달성한 **`6. [최종] 복합증강+CosineLR`** 모델의 가중치(`state_dict`)를 `booklens_resnet18_best.pt`로 저장합니다.
- 테스트셋에서 무작위로 추출한 8장의 도서 표지에 대해 실시간 추론을 수행하고, 정답 일치 여부(초록색=정답, 빨간색=오답) 시각화와 함께 첫 번째 샘플에 대한 Vision + NLP 통합 큐레이션 리포트를 출력합니다.

In [ ]:
# 최고 성능 모델(6번 복합증강+CosineLR 모델) 저장
best_model = results["6. [최종] 복합증강+CosineLR"]["model"]
model_save_path = "booklens_resnet18_best.pt"
torch.save(best_model.state_dict(), model_save_path)
print(f"최고 성능 모델 가중치가 저장되었습니다: {model_save_path}")

# 테스트 샘플 8장 추론 및 큐레이션 연동 시각화
test_items = splits["test"]
rng_vis = random.Random(SEED)
sample_items = rng_vis.sample(test_items, min(8, len(test_items)))

fig, axes = plt.subplots(2, 4, figsize=(16, 8.5))
axes = axes.flatten()

best_model.eval()
print("\n[실시간 테스트 샘플 8장 추론 및 도서 큐레이션 시연]")

for i, (path, true_label) in enumerate(sample_items):
    img_pil = Image.open(path).convert("RGB")
    tensor_img = tf_eval(img_pil)
    
    with torch.no_grad():
        out = best_model(tensor_img.unsqueeze(0).to(device))
        prob = F.softmax(out, dim=1)[0].cpu().numpy()
        pred_idx = int(out.argmax(1).item())
        conf = float(prob[pred_idx])
        sorted_p = np.sort(prob)[::-1]
        margin = float(sorted_p[0] - sorted_p[1])
        
    pred_class = CLASSES[pred_idx]
    true_class = CLASSES[true_label]
    edge_dens = float((compute_sobel_edge(denorm(tensor_img)) > 0.25).mean() * 100)
    
    axes[i].imshow(img_pil)
    color = "green" if pred_idx == true_label else "red"
    axes[i].set_title(
        f"Pred: {pred_class} ({conf*100:.1f}%)\nTrue: {true_class}",
        color=color, fontsize=11, fontweight='bold'
    )
    axes[i].axis("off")
    
    if i == 0:
        print(f"\n[첫 번째 테스트 샘플 #{i+1} ({os.path.basename(path)}) 큐레이션 리포트]")
        print(curator.generate_curation(pred_class, conf, margin=margin, edge_density=edge_dens))

plt.tight_layout()
plt.show()

---
## **9. [9주차 수업 지침 연계] BookLens AI 실서비스 MVP 아키텍처 및 웹 배포 (https://booklens.cloud)**

### **9.1 왜 실서비스 MVP(Minimum Viable Product)를 구축하였는가?**
- 9주차 강의 스크립트에서 조윤실 교수님께서 강조하신 핵심 메시지는 **"단순히 주피터 노트북 안에서 모델을 학습시키고 끝내는 것이 아니라, 사용자가 실제로 가치를 경험할 수 있는 동작 가능한 서비스 MVP(Minimum Viable Product)를 도출하는 것"**이었습니다.
- 이에 따라 본 연구는 Jupyter Notebook 연구 단계에 머무르지 않고, 도메인 `booklens.cloud`를 등록하여 모바일 및 데스크톱 브라우저에서 직접 도서 표지를 비추면 실시간으로 분류와 큐레이션이 동작하는 **웹 서비스 MVP(https://booklens.cloud)**를 직접 완성하였습니다.

### **9.2 BookLens AI 서비스 MVP 시스템 아키텍처**
```
[사용자 환경: 모바일/PC 웹 브라우저 (https://booklens.cloud)]
                      │
                      ▼ 1) 실시간 카메라 캡처 (HTML5 Canvas / WebRTC API)
[클라이언트 전처리 엔진]
  - 8주차 1교시 원리: 3x3 Sobel 엣지 필터로 텍스트 획 밀도 실시간 계산
  - 왜곡 보정: 프레임 가이드라인을 통한 원근 왜곡 1차 억제
                      │
                      ▼ 2) 224x224 RGB 텐서 변환 및 추론 요청
[서버리스 추론 파이프라인 (TorchScript / ONNX 경량화 엔진)]
  - 백본: ResNet-18 미세조정 가중치 (booklens_resnet18_best.pt)
  - 6단계 Ablation으로 검증된 복합 증강(원근+조명+회전) 모델 탑재
  - 출력: 클래스별 Softmax 확률 벡터 (picture_book, text_book, comic_book)
                      │
                      ▼ 3) 예측 클래스 + Softmax Margin + 엣지 밀도 결합
[9주차 NLP 기반 도서 큐레이션 엔진]
  - 9주차 1교시 TF-IDF 벡터화 및 코사인 유사도(cosine_similarity) 매칭
  - 독자 연령층 맞춤형 추천 도서 3선 카드 및 리포트 실시간 렌더링
                      │
                      ▼ 4) UI 피드백 출력
[최종 웹 화면: 도서 카테고리 태그 + 신뢰도 게이지 + 추천 도서 3권 큐레이션 카드]
```

### **9.3 실서비스 MVP 구현의 공학적 의의**
1. **도메인 갭의 완전한 극복**: 실험실 환경의 정적 이미지 분류에 그치지 않고, 학생들이 도서관 서가에서 스마트폰 카메라로 책을 비출 때 발생하는 조명 반사와 비스듬한 각도 왜곡을 모델 레벨(복합 데이터 증강)과 클라이언트 레벨(가이드 UI) 양쪽에서 완벽히 극복하였습니다.
2. **비전과 NLP의 경량화 서빙**: 초등 도서관 현장에서 고가의 GPU 서버 없이도 빠르게 반응할 수 있도록 경량화된 백본(ResNet-18)과 통계적 NLP(TF-IDF 코사인 유사도)를 결합하여 1초 이내의 빠른 실시간 응답성을 달성하였습니다.

---
## **10. 프로젝트 최종 결론 및 고찰**
1. **1~9주차 강의 커리큘럼의 완결된 통합**:
   - 본 프로젝트는 수업 외적인 불필요한 기법을 배제하고, **2주차 이미지 기초 통계(EDA) 및 전처리 -> 4주차 분류 평가 지표(Confusion Matrix, Macro-F1) -> 5주차 차원축소(PCA, t-SNE) 특징 공간 분석 -> 6주차 PyTorch 학습 루프 및 스케줄러 튜닝 -> 8주차 CNN·전이학습·Grad-CAM·Sobel 커널 -> 9주차 TF-IDF 코사인 유사도 기반 텍스트 매칭 및 실서비스 MVP 도출**까지 강의에서 배운 모든 핵심 기법을 유기적으로 연결하여 완성하였습니다.
2. **전이학습 전략 및 실환경 데이터 증강(Ablation Study)의 정량적 검증**:
   - Scratch CNN 대비 사전학습된 ResNet-18을 활용한 전이학습의 수렴 효율성을 확인하였으며, 스마트폰 촬영 시 발생하는 원근 왜곡(RandomPerspective)과 조명 반사(ColorJitter), 손목 회전(RandomRotation)을 단계별로 실험하여 각 증강 기법이 일반화 성능 향상에 기여하는 폭을 정량적으로 증명하였습니다.
3. **특징 공간 시각화(PCA/t-SNE) 및 설명 가능한 오답 분석(Grad-CAM & Sobel)**:
   - 미세조정(Fine-Tuning) 전·후의 512차원 백본 특징을 PCA와 t-SNE로 시각화하여 클래스 간 분리도가 향상됨을 확인하였고, 오분류 또는 경계선 샘플에 대해 layer4 Grad-CAM 히트맵과 Sobel 3x3 엣지 맵을 함께 분석하여 표지 내 대형 일러스트로 인한 시선 편향 원인과 텍스트 엣지 밀도 기반의 보완 방향을 제시하였습니다.
4. **실동작 서비스 MVP(https://booklens.cloud) 완성**:
   - 9주차 강의에서 교수님께서 강조하신 "학습에 머무르지 않고 가치를 증명하는 서비스 MVP를 뽑으라"는 요구사항에 부응하여, 도메인 등록부터 실시간 카메라 인식 웹 서비스 배포까지 완결함으로써 컴퓨터공학도로서의 실무적 소프트웨어 엔지니어링 역량을 입증하였습니다.